# Analise Bibliometrica com Semantic Scholar

Notebook pronto para Google Colab com busca na API da Semantic Scholar, tabelas bibliometricas, palavras-chave, bigramas, autores, areas e exportacao em CSV/Excel.


## Como usar

1. Rode a celula de instalacao.
2. Preencha a consulta e, se quiser, informe sua `S2_API_KEY` na celula de parametros.
3. Execute a busca.
4. Veja os graficos, tabelas e exporte os resultados.


In [ ]:
!pip -q install pandas requests matplotlib openpyxl

In [ ]:
import io
import re
from collections import Counter
from dataclasses import dataclass
from typing import Any, Iterable

import matplotlib.pyplot as plt
import pandas as pd
import requests
from getpass import getpass
from google.colab import files
from requests import Session
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

SEMANTIC_SCHOLAR_BULK_URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
FIELDS = ",".join([
    "title",
    "abstract",
    "year",
    "publicationDate",
    "citationCount",
    "influentialCitationCount",
    "referenceCount",
    "venue",
    "publicationTypes",
    "fieldsOfStudy",
    "authors",
    "openAccessPdf",
    "url",
    "externalIds",
])

STOPWORDS = {
    "a", "about", "analysis", "and", "are", "artificial", "as", "at", "also", "among", "approach",
    "article", "based", "be", "been", "being", "between", "biblioteca", "bibliotecas", "can", "com",
    "could", "da", "das", "de", "design", "do", "dos", "e", "education", "em", "estudo", "for",
    "from", "has", "how", "in", "intelligence", "into", "its", "library", "libraries", "machine",
    "main", "many", "may", "methods", "more", "most", "na", "nas", "new", "no", "nos", "o",
    "of", "on", "one", "or", "our", "paper", "para", "por", "purpose", "research", "results",
    "review", "scholar", "semantic", "shows", "study", "systematic", "such", "than", "that", "the",
    "their", "them", "there", "these", "this", "those", "to", "two", "um", "uma", "use", "used",
    "using", "various", "was", "were", "what", "when", "where", "which", "while", "whose", "will",
    "with", "within", "without", "would"
}


@dataclass
class SearchResult:
    papers: list[dict[str, Any]]
    total_results: int
    retrieved_results: int


def build_headers(api_key: str) -> dict[str, str]:
    headers = {"Accept": "application/json"}
    if api_key.strip():
        headers["x-api-key"] = api_key.strip()
    return headers


def build_session() -> Session:
    retry = Retry(
        total=5,
        backoff_factor=1.5,
        status_forcelist=[429, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        respect_retry_after_header=True,
    )
    session = Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


def safe_int(value: Any) -> int:
    try:
        return int(value)
    except (TypeError, ValueError):
        return 0


def safe_join(values: Iterable[Any]) -> str:
    cleaned = []
    for value in values:
        if value is None:
            continue
        text = str(value).strip()
        if text and text not in cleaned:
            cleaned.append(text)
    return "; ".join(cleaned)


def extract_unique_parts(value: Any) -> list[str]:
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except TypeError:
        pass
    parts = []
    for part in str(value).split(";"):
        cleaned = part.strip()
        if cleaned and cleaned not in parts:
            parts.append(cleaned)
    return parts


def tokenize_text(text: str, min_length: int = 3) -> list[str]:
    tokens = []
    for token in re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]{3,}", str(text).lower()):
        if len(token) >= min_length and token not in STOPWORDS:
            tokens.append(token)
    return tokens


def top_terms(texts: pd.Series, n: int = 20) -> pd.DataFrame:
    counter = Counter()
    for text in texts.dropna().astype(str):
        for token in tokenize_text(text):
            counter[token] += 1
    return pd.DataFrame(counter.most_common(n), columns=["termo", "frequencia"])


def top_ngrams(texts: pd.Series, n: int = 20, size: int = 2) -> pd.DataFrame:
    counter = Counter()
    for text in texts.dropna().astype(str):
        tokens = tokenize_text(text)
        for idx in range(len(tokens) - size + 1):
            gram = " ".join(tokens[idx:idx + size])
            counter[gram] += 1
    return pd.DataFrame(counter.most_common(n), columns=[f"{size}grama", "frequencia"])


def fetch_bulk_page(session: Session, api_key: str, query: str, year_filter: str, page_size: int, token: str | None = None) -> dict[str, Any]:
    params = {"query": query, "fields": FIELDS, "limit": page_size}
    if year_filter.strip():
        params["year"] = year_filter.strip()
    if token:
        params["token"] = token

    response = session.get(
        SEMANTIC_SCHOLAR_BULK_URL,
        headers=build_headers(api_key),
        params=params,
        timeout=45,
    )
    response.raise_for_status()
    return response.json()


def fetch_all_results(api_key: str, query: str, year_filter: str, page_size: int, max_results: int) -> SearchResult:
    session = build_session()
    papers = []
    token = None
    total_results = 0

    while len(papers) < max_results:
        payload = fetch_bulk_page(session, api_key, query, year_filter, page_size, token)
        if not total_results:
            total_results = safe_int(payload.get("total"))

        page_papers = payload.get("data") or []
        if not page_papers:
            break

        remaining = max_results - len(papers)
        papers.extend(page_papers[:remaining])
        if len(papers) >= max_results:
            break

        token = payload.get("token")
        if not token:
            break

    deduped = {}
    for idx, paper in enumerate(papers):
        dedupe_key = str(paper.get("paperId") or f"row-{idx}")
        deduped[dedupe_key] = paper

    final_papers = list(deduped.values())[:max_results]
    return SearchResult(final_papers, total_results, len(final_papers))


def papers_to_dataframe(papers: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for paper in papers:
        authors = paper.get("authors") or []
        external_ids = paper.get("externalIds") or {}
        open_access_pdf = paper.get("openAccessPdf") or {}
        rows.append({
            "paper_id": paper.get("paperId"),
            "titulo": paper.get("title"),
            "resumo": paper.get("abstract"),
            "primeiro_autor": authors[0].get("name") if authors else None,
            "autores": safe_join(author.get("name") for author in authors),
            "ano": paper.get("year"),
            "data_publicacao": paper.get("publicationDate"),
            "periodico_venue": paper.get("venue"),
            "tipos_publicacao": safe_join(paper.get("publicationTypes") or []),
            "areas_conhecimento": safe_join(paper.get("fieldsOfStudy") or []),
            "citacoes": safe_int(paper.get("citationCount")),
            "citacoes_influentes": safe_int(paper.get("influentialCitationCount")),
            "referencias": safe_int(paper.get("referenceCount")),
            "doi": external_ids.get("DOI"),
            "corpus_id": external_ids.get("CorpusId"),
            "url": paper.get("url"),
            "status_acesso_aberto": open_access_pdf.get("status"),
            "pdf_acesso_aberto": open_access_pdf.get("url"),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
    df["tem_pdf_aberto"] = df["pdf_acesso_aberto"].fillna("").astype(str).str.strip().ne("")
    df["texto_analise"] = (
        df["titulo"].fillna("").astype(str).str.strip() + " " + df["resumo"].fillna("").astype(str).str.strip()
    ).str.strip()
    return df.sort_values(["citacoes", "ano"], ascending=[False, False], na_position="last").reset_index(drop=True)


def filter_dataframe(df: pd.DataFrame, min_citations: int = 0, only_open_access: bool = False) -> pd.DataFrame:
    filtered = df.copy()
    if min_citations > 0:
        filtered = filtered[filtered["citacoes"] >= min_citations]
    if only_open_access:
        filtered = filtered[filtered["tem_pdf_aberto"] | filtered["status_acesso_aberto"].fillna("").eq("OPEN")]
    return filtered.reset_index(drop=True)


def build_year_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    summary = (
        df.dropna(subset=["ano"])
        .groupby("ano", as_index=False)
        .agg(documentos=("paper_id", "nunique"), citacoes=("citacoes", "sum"), pdf_aberto=("tem_pdf_aberto", "sum"))
        .sort_values("ano")
    )
    if not summary.empty:
        summary["citacoes_medias"] = (summary["citacoes"] / summary["documentos"]).round(2)
    return summary


def build_entity_summary(df: pd.DataFrame, column_name: str, label: str, top_n: int = 20) -> pd.DataFrame:
    rows = []
    for row in df.itertuples(index=False):
        for entity in extract_unique_parts(getattr(row, column_name)):
            rows.append({label: entity, "paper_id": getattr(row, "paper_id"), "citacoes": safe_int(getattr(row, "citacoes"))})
    if not rows:
        return pd.DataFrame()
    summary = (
        pd.DataFrame(rows)
        .groupby(label, as_index=False)
        .agg(documentos=("paper_id", "nunique"), citacoes=("citacoes", "sum"))
        .sort_values(["documentos", "citacoes"], ascending=[False, False])
        .head(top_n)
        .reset_index(drop=True)
    )
    return summary


def build_keyword_table(df: pd.DataFrame, text_column: str, top_n: int = 20) -> pd.DataFrame:
    term_counter = Counter()
    citation_counter = Counter()
    doc_counter = Counter()
    for row in df.itertuples(index=False):
        tokens = set(tokenize_text(getattr(row, text_column)))
        citations = safe_int(getattr(row, "citacoes"))
        for token in tokens:
            term_counter[token] += 1
            citation_counter[token] += citations
            doc_counter[token] += 1
    rows = [
        {
            "palavra_chave": term,
            "documentos": doc_counter[term],
            "citacoes": citation_counter[term],
            "citacoes_medias": round(citation_counter[term] / doc_counter[term], 2),
        }
        for term in term_counter
    ]
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["documentos", "citacoes"], ascending=[False, False]).head(top_n).reset_index(drop=True)


def dataframe_to_excel_bytes(df: pd.DataFrame) -> bytes:
    buffer = io.BytesIO()
    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="documentos", index=False)
        build_year_summary(df).to_excel(writer, sheet_name="anos", index=False)
        build_entity_summary(df, "autores", "autor", top_n=50).to_excel(writer, sheet_name="autores", index=False)
        build_entity_summary(df, "areas_conhecimento", "area", top_n=50).to_excel(writer, sheet_name="areas", index=False)
        top_terms(df["texto_analise"], n=50).to_excel(writer, sheet_name="palavras", index=False)
        top_ngrams(df["texto_analise"], n=40, size=2).to_excel(writer, sheet_name="bigramas", index=False)
        build_keyword_table(df, "texto_analise", top_n=50).to_excel(writer, sheet_name="palavras_citacoes", index=False)
    return buffer.getvalue()


In [ ]:
#@title Parametros da busca
QUERY = '"artificial intelligence" libraries' #@param {type:"string"}
YEAR_FILTER = '2020-' #@param {type:"string"}
PAGE_SIZE = 50 #@param {type:"integer"}
MAX_RESULTS = 500 #@param {type:"integer"}
MIN_CITATIONS = 0 #@param {type:"integer"}
ONLY_OPEN_ACCESS = False #@param {type:"boolean"}

print("Voce pode aumentar MAX_RESULTS para 10000 ou 20000, mas consultas grandes demoram mais e usam mais a API.")
print("Se voce tiver chave da API, informe abaixo. Se nao tiver, pode deixar em branco.")
S2_API_KEY = getpass("S2_API_KEY: ")


In [ ]:
result = fetch_all_results(
    api_key=S2_API_KEY,
    query=QUERY,
    year_filter=YEAR_FILTER,
    page_size=PAGE_SIZE,
    max_results=MAX_RESULTS,
)

df = papers_to_dataframe(result.papers)
filtered_df = filter_dataframe(df, min_citations=MIN_CITATIONS, only_open_access=ONLY_OPEN_ACCESS)

print({
    "total_estimado_api": result.total_results,
    "documentos_recuperados": result.retrieved_results,
    "documentos_filtrados": len(filtered_df),
})
display(filtered_df.head(10))


In [ ]:
year_summary = build_year_summary(filtered_df)
author_summary = build_entity_summary(filtered_df, "autores", "autor", top_n=10)
field_summary = build_entity_summary(filtered_df, "areas_conhecimento", "area", top_n=10)
keyword_summary = top_terms(filtered_df["texto_analise"], n=15)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

if not year_summary.empty:
    axes[0, 0].bar(year_summary["ano"].astype(str), year_summary["documentos"], color="#1D4E89")
    axes[0, 0].set_title("Publicacoes por ano")
    axes[0, 0].tick_params(axis="x", rotation=45)

if not author_summary.empty:
    axes[0, 1].barh(author_summary["autor"], author_summary["documentos"], color="#2A9D8F")
    axes[0, 1].set_title("Top autores")

if not field_summary.empty:
    axes[1, 0].barh(field_summary["area"], field_summary["documentos"], color="#E76F51")
    axes[1, 0].set_title("Assuntos mais frequentes")

if not keyword_summary.empty:
    axes[1, 1].barh(keyword_summary["termo"], keyword_summary["frequencia"], color="#F4A261")
    axes[1, 1].set_title("Palavras-chave mais frequentes")

plt.tight_layout()
plt.show()


In [ ]:
top_cited = filtered_df[["titulo", "primeiro_autor", "ano", "citacoes", "url"]].head(15)
keyword_table = build_keyword_table(filtered_df, "texto_analise", top_n=20)
bigram_summary = top_ngrams(filtered_df["texto_analise"], n=20, size=2)

print("\nArtigos mais citados")
display(top_cited)

print("\nPalavras-chave com mais impacto")
display(keyword_table)

print("\nExpressoes mais frequentes")
display(bigram_summary)


In [ ]:
csv_name = "semantic_scholar_bibliometria_colab.csv"
xlsx_name = "semantic_scholar_bibliometria_colab.xlsx"

filtered_df.to_csv(csv_name, index=False)
with open(xlsx_name, "wb") as fp:
    fp.write(dataframe_to_excel_bytes(filtered_df))

print(f"Arquivos gerados: {csv_name} e {xlsx_name}")
print("Descomente as linhas abaixo se quiser baixar automaticamente no Colab.")
# files.download(csv_name)
# files.download(xlsx_name)
